<a href="https://colab.research.google.com/github/charlie-aashutosh/100-Next-Gen-Engg-C4/blob/main/RAG_with_Gemini_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-core langchain-google-genai faiss-cpu #faiss is the vectordb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 kB 23.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 114.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.5 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninsta

In [2]:
import os
import getpass

os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your Gemini API Key")

Enter your Gemini API Key··········


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

In [4]:
!pip install pypdf #pypdf helps in loading and reading a dataset

In [5]:
from langchain_community.document_loaders import PyPDFLoader

In [6]:
loader=PyPDFLoader("/content/My_AxiOraa_Ltd_Synthetic_Annual_Report_2025.pdf")

In [7]:
documents=loader.load()
report=""

for page in documents:
  report+=page.page_content+'\n'

print("Total number of pages",len(documents))
print(report[:500]) #first 500 tokens from the annual report pdf

Total number of pages 9
AxiOraa Ltd.
Synthetic Annual Report 2025
About the Company
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrati


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,  #each chunk will have 1000 tokens
    chunk_overlap=100, #overlap of 100 tokens for each chunk created
)

In [9]:
chunk=text_splitter.split_documents(documents)
print(len(chunk)) #total number of chunks created is 42 based on 1000 tokens in each chunk and 100 tokens overlap

42


In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [11]:
embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [12]:
vector_store=FAISS.from_documents(
    documents=chunk,
    embedding=embedding
)

In [13]:
query="What investments did AxiOraa make in Artificial Intelligence"

In [18]:
result=vector_store.similarity_search_with_score(query)

for i, (doc, score) in enumerate(result):
  print(f"RESULT{i}")
  print(f"Document: {doc.page_content[:500]}")
  print(f"Score: {score}")

RESULT0
Document: management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manufacturing and public sector clients globally. It delivers AI strategy, data
engineering, cloud modernization and GenAI solutions. This section includes narrati
Score: 0.5133719444274902
RESULT1
Document: engineering, cloud modernization and GenAI solutions. This section includes narrative analysis,
management commentary, illustrative metrics, assumptions, trends, benchmark observations, and
fictional disclosures suitable for AI training demonstrations. Figures are synthetic and intended
solely for educational purposes.
AxiOraa Ltd. is a fictional AI-powered consulting and analytics company serving banking,
insurance, healthcare, manu

In [28]:
retriever=vector_store.as_retriever(
    search_type="similarity", #similarity search takes place
    search_kwargs={"k":1} #most appropriate response gets retrieved
)

In [29]:
from langchain_core.prompts import ChatPromptTemplate

In [30]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are Senior Management Consultant and your job is to
    answer the user questions based on the retrieved context only.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context: {context}
    Question: {question}

    Provide the following:
    1. Answer:
    2. Supporting evidence
    """
)

In [31]:
from langchain_core.output_parsers import StrOutputParser

In [32]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [33]:
question="help me understand the financial and operational highlights of this company"

In [34]:
retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke(
    {
        "context":context,
        "question":question
    }
)

In [35]:
print(response)

**1. Answer:**

Here is a summary of the financial and operational highlights for the company based on the provided data:

**Operational Highlights:**
* **Service Offerings:** The company operates across four primary service pillars: Strategy Consulting, AI Engineering, Managed Analytics, and Intelligent Automation.
* **Geographic Reach:** Its major operating markets span APAC (Asia-Pacific), Europe, and North America.

**Financial Highlights:**
* **Revenue:** Expanded by 21.1%, growing from USD 2.18 billion to USD 2.64 billion.
* **Gross Margin:** Profitability at the gross level improved by 2.7 percentage points, moving from 41.5% to 44.2%.
* **EBITDA Margin:** Achieved a solid EBITDA margin of 22.8%.
* **Net Profit:** Increased to USD 352 million.
* **Cash Flow:** Maintained a strong Free Cash Flow position.

*(Note: Per the commentary, these figures are synthetic and intended for educational/training purposes).*

***

**2. Supporting evidence:**

* **Operational footprint & markets

In [36]:
question="how is the company doing with regards to the AI Investments?"

In [37]:
retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response=rag_chain.invoke(
    {
        "context":context,
        "question":question
    }
)

In [38]:
print(response)

1. **Answer:**
The company has made significant investments in LLMOps, vector databases, evaluation frameworks, LangChain accelerators, and AI observability. 

2. **Supporting evidence:** 
"Significant investments were made in LLMOps, vector databases, evaluation frameworks, LangChain accelerators and AI observability."
